In [1]:
from typing import TypedDict,Annotated
from langgraph.types import Send
import operator
from langgraph.graph import START,END,StateGraph,MessagesState
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

True

In [5]:
class GlobalState(TypedDict):
    items: list
    results: Annotated[list,operator.add]

class ItemState(TypedDict):
    item: str

def process_state(state:ItemState)->dict:
    item=state['item']
    print(f"Processing Data : {state['item']}")
    result=f"Processed:{state['item']}"
    return {"results": [result]}

def distribute_load(state:GlobalState)->dict:
    print(f"Distributing {state['items']} loads for processing")
    return [Send("process",{"item":item}) for item in state['items']]

graph=StateGraph(GlobalState)
graph.add_node("process",process_state)
graph.add_conditional_edges(START,distribute_load,["process"])
graph.add_edge("process",END)

app=graph.compile()
result=app.invoke({
    "items":["apple", "banana", "cherry", "date"],
    "results":[]
})

print(result)


Distributing ['apple', 'banana', 'cherry', 'date'] loads for processing
Processing Data : apple
Processing Data : banana
Processing Data : cherry
Processing Data : date
{'items': ['apple', 'banana', 'cherry', 'date'], 'results': ['Processed:apple', 'Processed:banana', 'Processed:cherry', 'Processed:date']}
